In [ ]:
import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep
)

df = pd.read_csv("../data/session_level_analysis.csv")

# 1. 汇总两组样本量和跳出数量
summary = (
    df.groupby("landing_page")
      .agg(
          sessions=("website_session_id", "size"),
          bounced_sessions=("bounced", "sum")
      )
      .reindex(["/home", "/lander-1"])
)

summary["bounce_rate"] = (
    summary["bounced_sessions"] / summary["sessions"]
)
print(summary)

# 2. 检查是否符合预期的50/50分流
total = summary["sessions"].sum()

srm = chisquare(
    f_obs=summary["sessions"].to_numpy(),
    f_exp=[total / 2, total / 2]
)
print(f"\nSRM p值：{srm.pvalue:.4f}")

# 3. 提取A、B两组数据
rate_a = summary.loc["/home", "bounce_rate"]
rate_b = summary.loc["/lander-1", "bounce_rate"]

# 4. 两比例检验：比较B与A的跳出率
z_stat, p_value = proportions_ztest(
    count=[bounce_b, bounce_a],
    nobs=[n_b, n_a],
    alternative="two-sided"
)

# 5. B-A差异的95%置信区间
ci_low, ci_high = confint_proportions_2indep(
    count1=bounce_b,
    nobs1=n_b,
    count2=bounce_a,
    nobs2=n_a,
    compare="diff",
    method="newcomb",
    alpha=0.05
)

print(f"A组跳出率：{rate_a:.2%}")
print(f"B组跳出率：{rate_b:.2%}")
print(f"B-A差异：{(rate_b - rate_a) * 100:.2f}个百分点")
print(
    f"95%置信区间："
    f"[{ci_low * 100:.2f}, {ci_high * 100:.2f}]个百分点"
)
print(f"双侧检验p值：{p_value:.6f}")

              sessions  bounced_sessions  bounce_rate
landing_page                                         
/home             2261              1319     0.583370
/lander-1         2315              1232     0.532181

SRM p值：0.4247
A组跳出率：58.34%
B组跳出率：53.22%
B-A差异：-5.12个百分点
95%置信区间：[-7.98, -2.24]个百分点
双侧检验p值：0.000491


In [2]:
purchase_summary = (
    df.groupby("landing_page")
      .agg(
          sessions=("website_session_id", "size"),
          purchased_sessions=("purchased", "sum")
      )
      .reindex(["/home", "/lander-1"])
)

purchase_summary["purchase_rate"] = (
    purchase_summary["purchased_sessions"]
    / purchase_summary["sessions"]
)

print(purchase_summary)

              sessions  purchased_sessions  purchase_rate
landing_page                                             
/home             2261                  72       0.031844
/lander-1         2315                  94       0.040605


In [3]:
purchase_a = int(
    purchase_summary.loc["/home", "purchased_sessions"]
)
purchase_b = int(
    purchase_summary.loc["/lander-1", "purchased_sessions"]
)

purchase_rate_a = purchase_a / n_a
purchase_rate_b = purchase_b / n_b

_, purchase_p = proportions_ztest(
    count=[purchase_b, purchase_a],
    nobs=[n_b, n_a],
    alternative="two-sided"
)

purchase_low, purchase_high = confint_proportions_2indep(
    count1=purchase_b,
    nobs1=n_b,
    count2=purchase_a,
    nobs2=n_a,
    compare="diff",
    method="newcomb",
    alpha=0.05
)

print(f"A组购买转化率：{purchase_rate_a:.2%}")
print(f"B组购买转化率：{purchase_rate_b:.2%}")
print(
    f"B-A差异："
    f"{(purchase_rate_b - purchase_rate_a) * 100:.2f}个百分点"
)
print(
    f"95%置信区间："
    f"[{purchase_low * 100:.2f}, "
    f"{purchase_high * 100:.2f}]个百分点"
)
print(f"双侧检验p值：{purchase_p:.6f}")

A组购买转化率：3.18%
B组购买转化率：4.06%
B-A差异：0.88个百分点
95%置信区间：[-0.21, 1.97]个百分点
双侧检验p值：0.113056
